# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

Run this notebook from `markdown_generator/` (or from the repo root with `uv run`), via:

```bash
jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```

The repo-root `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to write data, HTML, and Javascript into the repo-root `talkmap/` folder.

In [ ]:
from pathlib import Path

import frontmatter
import getorg
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut

In [ ]:
# Resolve paths relative to this notebook's directory (markdown_generator/)
ROOT = Path.cwd()
if ROOT.name == "markdown_generator":
    ROOT = ROOT.parent
elif (ROOT / "markdown_generator").is_dir():
    pass
else:
    raise FileNotFoundError("Run this notebook from the repo root or markdown_generator/")

TALKS_DIR = ROOT / "_talks"
OUTPUT_DIR = ROOT / "talkmap"
g = sorted(TALKS_DIR.glob("*.md"))
print(f"Found {len(g)} talk files in {TALKS_DIR}")

In [ ]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
geocoder = Nominatim(user_agent="academicpages.github.io")
location_dict = {}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [ ]:
# Perform geolocation
for file in g:
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description
    title = data['title'].strip()
    venue = data['venue'].strip()
    location = data['location'].strip()
    description = f"{title}<br />{venue}; {location}"

    # Geocode the location and report the status
    try:
        location_dict[description] = geocoder.geocode(location, timeout=TIMEOUT)
        print(description, location_dict[description])
    except ValueError as ex:
        print(f"Error: geocode failed on input {location} with message {ex}")
    except GeocoderTimedOut as ex:
        print(f"Error: geocode timed out on input {location} with message {ex}")
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {location} with message {ex}")

In [ ]:
# Save the map into the repo-root talkmap/ folder
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(
    location_dict, folder_name=str(OUTPUT_DIR), hashed_usernames=False
)